In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')


import numpy as np
import pandas as pd
import cv2
from pathlib import Path

REPLAY_PATH = ROOT/'a5_v1_freeze'/'frozen'/'A5-v1-print-p0980-margin0960-run2'/'826035ea9745644a0b21b10ec1fc3d1b08670f65f2a5537e1116f4428f1a9f59'/'replay'/'Page003'
NPZ = ROOT/'observable_inputs.npz'
OUT = ROOT/'page003_replay_export'
OUT.mkdir(exist_ok=True)

with np.load(NPZ) as z:
    print("Arrays:", z.files)

    for name in z.files:
        a = z[name]
        print(name, a.shape, a.dtype, a.min(), a.max())

        # Byte-valued grayscale arrays
        if (
            a.ndim == 2
            and a.dtype == np.uint8
        ):
            cv2.imwrite(str(OUT / f"{name}.png"), a)

        # Boolean/binary maps
        elif (
            a.ndim == 2
            and (
                a.dtype == bool
                or np.array_equal(a, a.astype(bool))
            )
        ):
            cv2.imwrite(
                str(OUT / f"{name}.png"),
                a.astype(np.uint8) * 255
            )

        # Everything else stays numeric
        else:
            pd.DataFrame(a).to_csv(
                OUT / f"{name}.csv",
                index=False,
                header=False
            )

for name in [
    "expected_probability",
    "expected_provisional",
    "expected_final",
]:
    a = np.load(f"{REPLAY_PATH}/{name}.npy", allow_pickle=True)

    pd.DataFrame({
        name: a
    }).to_csv(
        f"{OUT}/{name}.csv",
        index=False
    )

print("Exported to:", OUT)